In [3]:
import torch
import torchvision
from torch.utils.data import DataLoader
from ppq import *
from ppq.api import *


      ____  ____  __   ____                    __              __
     / __ \/ __ \/ /  / __ \__  ______ _____  / /_____  ____  / /
    / /_/ / /_/ / /  / / / / / / / __ `/ __ \/ __/ __ \/ __ \/ /
   / ____/ ____/ /__/ /_/ / /_/ / /_/ / / / / /_/ /_/ / /_/ / /
  /_/   /_/   /_____\___\_\__,_/\__,_/_/ /_/\__/\____/\____/_/




In [ ]:
DEVICE = 'cuda'
PLATFORM = TargetPlatform.PPL_CUDA_INT8

def collate_fn(batch: torch.Tensor) -> torch.Tensor:
    return batch.to(DEVICE)

dataset = [torch.rand(size=[3, 224, 224]) for _ in range(1024)]

model = torchvision.models.mobilenet.mobilenet_v2(pretrained=True)
model = model.to(DEVICE)

quant_setting = QuantizationSettingFactory.pplcuda_setting()
quant_setting.advanced_optimization = True
calibaration_dataloader = DataLoader(dataset=dataset, batch_size=32)

quantized = quantize_torch_model(
    model=model, calib_dataloader=calibaration_dataloader,
    calib_steps=32, input_shape=[1, 3, 224, 224],
    setting=quant_setting, collate_fn=collate_fn, platform=PLATFORM,
    onnx_export_file='onnx.model', device=DEVICE, verbose=1)


assert isinstance(quantized, BaseGraph)

export_ppq_graph(graph=quantized, platform=PLATFORM,
                 graph_save_to='quantized.onnx',
                 config_save_to='quantized.json')

reports = graphwise_error_analyse(
    graph=quantized, running_device=DEVICE, collate_fn=collate_fn,
    dataloader=calibaration_dataloader)



/home/faione/miniconda3/envs/MLC/lib/python3.8/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/faione/miniconda3/envs/MLC/lib/python3.8/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


[14:09:41] PPQ Quantization Fusion Pass Running ...       Finished.
[14:09:41] PPQ Quantize Simplify Pass Running ...         Finished.
[14:09:41] PPQ Parameter Quantization Pass Running ...    Finished.
[14:09:42] PPQ Runtime Calibration Pass Running ...       

Calibration Progress(Phase 1): 100%|██████████| 32/32 [01:04<00:00,  2.02s/it]


Finished.
[14:10:48] PPQ Quantization Alignment Pass Running ...    Finished.
[14:10:48] PPQ Passive Parameter Quantization Running ... Finished.
[14:10:48] PPQ Parameter Baking Pass Running ...          Finished.
--------- Network Snapshot ---------
Num of Op:                    [100]
Num of Quantized Op:          [100]
Num of Variable:              [277]
Num of Quantized Var:         [277]
------- Quantization Snapshot ------
Num of Quant Config:          [386]
ACTIVATED:                    [45]
BAKED:                        [53]
OVERLAPPED:                   [145]
PASSIVE:                      [20]
PASSIVE_BAKED:                [123]
Network Quantization Finished.


Analysing Graphwise Quantization Error(Phrase 1):: 100%|██████████| 8/8 [00:00<00:00,  9.68it/s]
Analysing Graphwise Quantization Error(Phrase 2):: 100%|██████████| 8/8 [00:01<00:00,  7.54it/s]

Layer                                            | NOISE:SIGNAL POWER RATIO 
/features/features.17/conv/conv.0/conv.0.0/Conv: | ████████████████████ | 68.320%
/features/features.16/conv/conv.2/Conv:          | █████████████████    | 59.156%
/features/features.15/conv/conv.2/Conv:          | ████████████████     | 55.621%
/features/features.14/conv/conv.2/Conv:          | ███████████████      | 50.308%
/features/features.16/conv/conv.0/conv.0.0/Conv: | ████████████         | 42.294%
/features/features.16/conv/conv.1/conv.1.0/Conv: | ██████████           | 33.446%
/features/features.15/conv/conv.0/conv.0.0/Conv: | ████████             | 28.348%
/features/features.15/conv/conv.1/conv.1.0/Conv: | ███████              | 24.452%
/features/features.14/conv/conv.1/conv.1.0/Conv: | ███████              | 23.376%
/features/features.13/conv/conv.2/Conv:          | ██████               | 19.996%
/features/features.14/conv/conv.0/conv.0.0/Conv: | ██████               | 19.288%
/features/features.17

In [1]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.current_device())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0))
print(torch.version.cuda)

True
0
1
NVIDIA GeForce RTX 4070 Ti SUPER
12.4
